# Omo Forest Reserve Ecological Informatics Project

# Notebook 05
## Advanced Community Ecology and Statistical Analysis

---

### Project

**Ecological Informatics Assessment of Tropical Tree Communities in Omo Forest Reserve, Nigeria**

---

### Purpose

This notebook extends the ecological analyses performed in the previous notebooks by investigating community structure, diversity patterns, ecological similarity, and multivariate relationships among sampling plots using advanced community ecology techniques.

The analyses include:

- Data Quality Assessment
- Community Matrix Construction
- Alpha Diversity
- Beta Diversity
- Community Similarity
- Hierarchical Cluster Analysis
- Principal Component Analysis (PCA)
- Non-metric Multidimensional Scaling (NMDS)
- PERMANOVA
- Indicator Species Analysis
- Publication-ready Tables
- Publication-ready Figures

---

### Input Dataset

`tree_analysis_ready.csv`

---

### Output Directory

`outputs/`

---

### Author

Peter Ugege

Forestry Research Institute of Nigeria (FRIN)

---

**Last Updated:** July 2026

# Table of Contents

1. Load Libraries
2. Project Configuration
3. Load Dataset
4. Data Quality Assurance
5. Species × Plot Matrix
6. Alpha Diversity Comparison
7. Beta Diversity
8. Community Similarity
9. Hierarchical Cluster Analysis
10. Principal Component Analysis
11. NMDS
12. PERMANOVA
13. Indicator Species Analysis
14. Export Publication Tables
15. Export Publication Figures
16. Ecological Interpretation
17. Notebook Summary

In [17]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Scientific plotting style
plt.style.use("ggplot")
sns.set_context("talk")

print("✓ Core libraries successfully imported.")

✓ Core libraries successfully imported.


In [18]:
# =============================================================================
# Import Community Ecology Libraries
# =============================================================================

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

from skbio.diversity import alpha_diversity
from skbio.diversity import beta_diversity
from skbio.stats.distance import permanova

print("✓ Community ecology libraries successfully imported.")

✓ Community ecology libraries successfully imported.


In [21]:
# =============================================================================
# Project Configuration
# =============================================================================
project_root = Path.cwd().parent

PROJECT_NAME = "Omo Forest Ecological Informatics Project"

INPUT_FILE = r"C:/Peter/SDM_data/omo_ecodb/omo-ecological-informatics/data/processed/tree_analysis_ready.csv"

OUTPUT_DIR = project_root / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Input   : {INPUT_FILE}")
print(f"Output  : {OUTPUT_DIR.resolve()}")

print("\nOutput folders created successfully.")

Project : Omo Forest Ecological Informatics Project
Input   : C:/Peter/SDM_data/omo_ecodb/omo-ecological-informatics/data/processed/tree_analysis_ready.csv
Output  : C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\outputs

Output folders created successfully.


# Load Dataset

The cleaned and harmonized dataset generated from Notebook 04 is loaded for advanced community ecology analyses.

In [22]:
# =============================================================================
# Load Analysis-Ready Dataset
# =============================================================================

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(f"Cannot locate input dataset: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("DATASET SUCCESSFULLY LOADED")
print("=" * 70)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print("=" * 70)

DATASET SUCCESSFULLY LOADED
Rows    : 206
Columns : 23


In [23]:
# =============================================================================
# Preview Dataset
# =============================================================================

print("First Five Records\n")

display(df.head())

print("\n")

print("Last Five Records\n")

display(df.tail())

First Five Records



,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,...,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat,Taxonomic_Resolution,Family,Genus,Taxonomic_Status
0,Alstonia boonei,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,0.0027,1.1560,2.0400,1.5984,Buffer,Major River,Species,Apocynaceae,Alstonia,ACCEPTED
1,Blighia sapida,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,...,0.0068,2.8901,5.1020,3.9961,Buffer,Major River,Species,Sapindaceae,Blighia,ACCEPTED
2,Bombax buonopozense,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,...,0.0096,4.0462,5.1020,4.5741,Buffer,Major River,Species,Malvaceae,Bombax,ACCEPTED
3,Bosqueia angolensis,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,...,0.0055,2.3121,3.0612,2.6866,Buffer,Major River,Species,Moraceae,Bosqueia,SYNONYM
4,Celtis zenkeri,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,...,0.0068,2.8901,4.0816,3.4859,Buffer,Major River,Species,Cannabaceae,Celtis,ACCEPTED




Last Five Records



,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,...,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat,Taxonomic_Resolution,Family,Genus,Taxonomic_Status
201,Rauvolfia vomitoria,2.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,...,0.004138,3.33333,4.0,3.666667,Transition,Upland,Species,Apocynaceae,Rauvolfia,ACCEPTED
202,Ricinodendron heudelotii,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,0.004138,3.33333,6.0,4.666667,Transition,Upland,Species,Euphorbiaceae,Ricinodendron,ACCEPTED
203,Theobroma cacao,NaN,3.0,5.0,5.0,NaN,1.0,5.0,3.0,4.0,...,0.038621,31.11110,16.0,23.555560,Transition,Upland,Species,Malvaceae,Theobroma,ACCEPTED
204,Trema orientalis,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.002759,2.22222,2.0,2.111111,Transition,Upland,Species,Cannabaceae,Trema,SYNONYM
205,Uapaca heudelotii,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,0.001379,1.11111,2.0,1.555556,Transition,Upland,Species,Phyllanthaceae,Uapaca,ACCEPTED


In [24]:
# =============================================================================
# Dataset Structure
# =============================================================================

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

df.info()

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               206 non-null    object 
 1   Plot1                 66 non-null     float64
 2   Plot2                 75 non-null     float64
 3   Plot3                 72 non-null     float64
 4   Plot4                 77 non-null     float64
 5   Plot5                 56 non-null     float64
 6   Plot6                 97 non-null     float64
 7   Plot7                 86 non-null     float64
 8   Plot8                 76 non-null     float64
 9   Plot9                 88 non-null     float64
 10  Plot10                93 non-null     float64
 11  Frequency             206 non-null    int64  
 12  Individuals           206 non-null    int64  
 13  Density               206 non-null    float64
 14  Relative_Density      206 non-null    float64
 15  Rel

In [25]:
# =============================================================================
# List Dataset Variables
# =============================================================================

variables = pd.DataFrame({
    "No.": range(1, len(df.columns)+1),
    "Variable": df.columns,
    "Data Type": df.dtypes.astype(str)
})

display(variables)

,No.,Variable,Data Type
Species,1,Species,object
Plot1,2,Plot1,float64
Plot2,3,Plot2,float64
Plot3,4,Plot3,float64
Plot4,5,Plot4,float64
Plot5,6,Plot5,float64
Plot6,7,Plot6,float64
Plot7,8,Plot7,float64
Plot8,9,Plot8,float64
Plot9,10,Plot9,float64


In [27]:
# =============================================================================
# Summary Statistics
# =============================================================================

display(df.describe(include="all").transpose())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Species,206,87,Cleistopholis patens,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plot1,66.0,NaN,NaN,NaN,2.151515,1.970884,1.0,1.0,1.0,2.0,12.0
Plot2,75.0,NaN,NaN,NaN,1.826667,1.671489,1.0,1.0,1.0,2.0,10.0
Plot3,72.0,NaN,NaN,NaN,2.222222,1.847894,1.0,1.0,1.0,2.0,9.0
Plot4,77.0,NaN,NaN,NaN,2.532468,2.668453,1.0,1.0,1.0,3.0,14.0
Plot5,56.0,NaN,NaN,NaN,2.428571,2.470606,1.0,1.0,1.0,2.25,12.0
Plot6,97.0,NaN,NaN,NaN,1.680412,1.454493,1.0,1.0,1.0,2.0,9.0
Plot7,86.0,NaN,NaN,NaN,1.790698,1.439904,1.0,1.0,1.0,2.0,9.0
Plot8,76.0,NaN,NaN,NaN,1.973684,2.026318,1.0,1.0,1.0,2.0,12.0
Plot9,88.0,NaN,NaN,NaN,1.852273,1.797442,1.0,1.0,1.0,2.0,11.0


### Interpretation

The analysis-ready dataset has been successfully imported into the notebook. An initial inspection of the dataset provides an overview of its size, variables, and data types. At this stage, no modifications are made to the dataset; the objective is simply to confirm that the imported data match the outputs generated in Notebook 04.

The next section performs a comprehensive quality assessment to evaluate completeness, consistency, duplicate observations, missing values, and other characteristics that may influence subsequent community ecology analyses.

# 4. Data Quality Assessment

This section evaluates the integrity of the community dataset before multivariate ecological analyses.

The assessment includes:

- Missing values
- Duplicate species
- Community matrix validation
- Plot abundance verification
- Empty plots
- Species with zero abundance
- Dataset suitability for community ecology analyses

In [28]:
# =============================================================================
# Identify Community Matrix Columns
# =============================================================================

plot_columns = [col for col in df.columns if col.startswith("Plot")]

print("=" * 70)
print("Community Matrix")
print("=" * 70)
print(f"Number of plots detected : {len(plot_columns)}")
print(f"Plot columns            : {plot_columns}")

Community Matrix
Number of plots detected : 10
Plot columns            : ['Plot1', 'Plot2', 'Plot3', 'Plot4', 'Plot5', 'Plot6', 'Plot7', 'Plot8', 'Plot9', 'Plot10']


In [29]:
# =============================================================================
# Missing Value Assessment
# =============================================================================

missing = pd.DataFrame({
    "Variable": df.columns,
    "Missing Values": df.isnull().sum().values
})

missing["Percent"] = (
    missing["Missing Values"] / len(df) * 100
).round(2)

display(missing)

missing.to_csv(TABLE_DIR / "missing_values.csv", index=False)

,Variable,Missing Values,Percent
0,Species,0,0.00
1,Plot1,140,67.96
2,Plot2,131,63.59
3,Plot3,134,65.05
4,Plot4,129,62.62
5,Plot5,150,72.82
6,Plot6,109,52.91
7,Plot7,120,58.25
8,Plot8,130,63.11
9,Plot9,118,57.28


In [30]:
# =============================================================================
# Duplicate Species Check
# =============================================================================

duplicate_species = df["Species"].duplicated().sum()

print("=" * 70)
print("Duplicate Species")
print("=" * 70)
print(f"Duplicate species records : {duplicate_species}")

Duplicate Species
Duplicate species records : 119


In [31]:
# =============================================================================
# Validate Community Matrix
# =============================================================================

community = df[plot_columns]

display(community.head())

print(f"\nCommunity matrix dimensions : {community.shape}")

,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10
0,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0
1,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,1.0
2,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,NaN
3,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN
4,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,NaN



Community matrix dimensions : (206, 10)


In [32]:
# =============================================================================
# Species with Zero Abundance
# =============================================================================

zero_species = df.loc[
    community.sum(axis=1) == 0,
    "Species"
]

print("=" * 70)
print("Species with Zero Total Abundance")
print("=" * 70)
print(f"Number of species : {len(zero_species)}")

if len(zero_species) > 0:
    display(zero_species)

Species with Zero Total Abundance
Number of species : 0


In [33]:
# =============================================================================
# Empty Plot Check
# =============================================================================

plot_totals = community.sum(axis=0)

display(plot_totals)

empty_plots = plot_totals[plot_totals == 0]

print("\nEmpty plots :", len(empty_plots))

Plot1     142.0
Plot2     137.0
Plot3     160.0
Plot4     195.0
Plot5     136.0
Plot6     163.0
Plot7     154.0
Plot8     150.0
Plot9     163.0
Plot10    154.0
dtype: float64


Empty plots : 0


In [34]:
# =============================================================================
# Inspect Duplicate Species
# =============================================================================

duplicate_species_df = (
    df[df["Species"].duplicated(keep=False)]
    .sort_values("Species")
)

print(f"Duplicate records : {len(duplicate_species_df)}")

display(
    duplicate_species_df[
        ["Species", "Zone", "Habitat"]
    ].head(30)
)

Duplicate records : 168


,Species,Zone,Habitat
25,Albizia zygia,Buffer,Stream
174,Albizia zygia,Transition,Stream
0,Alstonia boonei,Buffer,Major River
55,Alstonia boonei,Buffer,Upland
103,Alstonia boonei,Core,Stream
26,Alstonia boonei,Buffer,Stream
175,Alstonia boonei,Transition,Stream
56,Anonidium mannii,Buffer,Upland
80,Anonidium mannii,Core,Major River
126,Anonidium mannii,Core,Upland


In [35]:
# =============================================================================
# Species Occurrence Frequency
# =============================================================================

species_counts = (
    df["Species"]
    .value_counts()
    .reset_index()
)

species_counts.columns = ["Species", "Records"]

display(species_counts.head(20))

,Species,Records
0,Cleistopholis patens,7
1,Ricinodendron heudelotii,7
2,Diospyros dendo,7
3,Strombosia pustulata,6
4,Sterculia rhinopetala,6
5,Blighia sapida,6
6,Alstonia boonei,5
7,Nesogordonia papaverifera,5
8,Lecaniodiscus cupanioides,5
9,Musanga cecropioides,5


In [36]:
# =============================================================================
# Replace Missing Plot Abundances with Zero
# =============================================================================

df[plot_columns] = (
    df[plot_columns]
    .fillna(0)
)

print("Missing abundance values replaced with zero.")

print(df[plot_columns].isna().sum())

Missing abundance values replaced with zero.
Plot1     0
Plot2     0
Plot3     0
Plot4     0
Plot5     0
Plot6     0
Plot7     0
Plot8     0
Plot9     0
Plot10    0
dtype: int64


In [37]:
# =============================================================================
# Explore Sampling Design
# =============================================================================

print("=" * 70)
print("Zones")
print("=" * 70)
print(df["Zone"].value_counts())

print()

print("=" * 70)
print("Habitats")
print("=" * 70)
print(df["Habitat"].value_counts())

print()

print("=" * 70)
print("Zone × Habitat")
print("=" * 70)

sampling_design = (
    df.groupby(["Zone", "Habitat"])
      .size()
      .reset_index(name="Species")
)

display(sampling_design)

Zones
Zone
Buffer        80
Core          68
Transition    58
Name: count, dtype: int64

Habitats
Habitat
Major River    73
Stream         68
Upland         65
Name: count, dtype: int64

Zone × Habitat


,Zone,Habitat,Species
0,Buffer,Major River,24
1,Buffer,Stream,31
2,Buffer,Upland,25
3,Core,Major River,23
4,Core,Stream,23
5,Core,Upland,22
6,Transition,Major River,26
7,Transition,Stream,14
8,Transition,Upland,18


In [38]:
community = df[plot_columns]

community.describe().T

,count,mean,std,min,25%,50%,75%,max
Plot1,206.0,0.689320,1.498156,0.0,0.0,0.0,1.0,12.0
Plot2,206.0,0.665049,1.335972,0.0,0.0,0.0,1.0,10.0
Plot3,206.0,0.776699,1.520154,0.0,0.0,0.0,1.0,9.0
Plot4,206.0,0.946602,2.036751,0.0,0.0,0.0,1.0,14.0
Plot5,206.0,0.660194,1.676543,0.0,0.0,0.0,1.0,12.0
Plot6,206.0,0.791262,1.302950,0.0,0.0,0.0,1.0,9.0
Plot7,206.0,0.747573,1.281907,0.0,0.0,0.0,1.0,9.0
Plot8,206.0,0.728155,1.553559,0.0,0.0,0.0,1.0,12.0
Plot9,206.0,0.791262,1.488204,0.0,0.0,0.0,1.0,11.0
Plot10,206.0,0.747573,1.128076,0.0,0.0,0.0,1.0,8.0
